# Lab 03 — LLM Calls: Single Input

**Knowledge base:** `knowledge_base/02_llms/01_how_llms_work.md`

**Concepts:** `generate_with_single_input` · roles · temperature · max_tokens · response parsing

This lab is entirely focused on `generate_with_single_input`.
You will run small experiments and observe exactly how each parameter affects the output.

---

In [1]:
import os, sys
sys.path.insert(0, ".")
from dotenv import load_dotenv
from utils import generate_with_single_input

load_dotenv(override=True)
print(f"✅ Backend: {os.environ.get('LLM_BACKEND', 'ollama')}")

✅ Backend: ollama


---
## 1 — The return value

Every call returns a dict with exactly two keys: `role` and `content`.
You always extract the answer from `output['content']`.

In [2]:
output = generate_with_single_input(prompt="What is the capital of France?")

# Inspect the full return value
print("Full return dict:")
print(output)
print()

# How to extract the answer in your own code
answer = output['content']
print(f"Answer: {answer}")

Full return dict:
{'role': 'assistant', 'content': 'The capital of France is **Paris**.'}

Answer: The capital of France is **Paris**.


---
## 2 — The role parameter

The role tells the model who is sending the message.
- `'user'` — a question or instruction (default)
- `'system'` — rules or a persona that shape the model's entire behaviour

In [ ]:
# role='user' (default) — a direct question
user_output = generate_with_single_input(
    prompt="What is 2 + 2?",
    role="user"
)
print("role='user':")
print(user_output['content'])

In [ ]:
# role='system' — gives the model a persona before any user question
# This is most useful in generate_with_multiple_input (lab04)
# but you can send a system message alone too
system_output = generate_with_single_input(
    prompt="You are a pirate assistant. Always answer in pirate style. What is 2 + 2?",
    role="user"
)
print("Pirate persona via prompt:")
print(system_output['content'])

---
## 3 — max_tokens

`max_tokens` limits how long the response can be.
Short values cut the answer off early. Long values allow full explanations.
Setting this to a sensible value avoids paying for unnecessarily long responses in production.

In [ ]:
prompt = "Explain what an LLM is in plain language."

for limit in [20, 80, 300]:
    out = generate_with_single_input(prompt=prompt, max_tokens=limit)
    print(f"\n── max_tokens={limit} ──")
    print(out['content'])

---
## 4 — temperature

`temperature` controls how predictable the output is.

- `0.0` → deterministic. The model always picks the most probable next token. Same input → same output every time.
- `1.0` → creative. The model samples from the distribution. Same input → different output each run.

**For RAG:** always use `temperature=0.0`. You want consistent, grounded answers, not creative variation.

In [ ]:
# Run this cell multiple times — the answer should not change
print("temperature=0.0 (deterministic) — run this cell 3 times:")
for i in range(3):
    out = generate_with_single_input(
        prompt="Name one country in North Africa.",
        temperature=0.0,
        max_tokens=15
    )
    print(f"  Run {i+1}: {out['content'].strip()}")

In [ ]:
# Run this cell multiple times — the answer may vary
print("temperature=1.0 (creative) — run this cell 3 times:")
for i in range(3):
    out = generate_with_single_input(
        prompt="Name one country in North Africa.",
        temperature=1.0,
        max_tokens=15
    )
    print(f"  Run {i+1}: {out['content'].strip()}")

---
## 5 — Calling the LLM multiple times in a loop

A common pattern: apply the same prompt template to a list of inputs.

In [ ]:
# Classify each document topic — a real RAG pre-processing step
documents = [
    "The Egyptian pound weakened against the dollar this quarter.",
    "NASA's Artemis mission targets a lunar landing in 2026.",
    "Barcelona defeated Real Madrid 3-1 in El Clásico.",
    "A new transformer architecture outperforms BERT on NLP benchmarks.",
]

def classify_topic(text: str) -> str:
    out = generate_with_single_input(
        prompt=f"Classify this text into one of: Finance, Space, Sports, Technology.\nText: {text}\nCategory:",
        max_tokens=10,
        temperature=0.0
    )
    return out['content'].strip()


print("Classifying documents:")
for doc in documents:
    topic = classify_topic(doc)
    print(f"  [{topic:12s}] {doc[:60]}...")

---
## 6 — Exercise

Write a function `answer_with_context` that:
1. Takes a `query: str` and a `context: str`
2. Builds an augmented prompt that includes both
3. Calls `generate_with_single_input` with `temperature=0.0`
4. Returns the answer string (not the full dict)

In [ ]:
def answer_with_context(query: str, context: str) -> str:
    """
    Call the LLM with a query grounded by a context string.

    Args:
        query:   The user's question.
        context: The text the LLM must use to answer.

    Returns:
        The LLM's answer as a plain string.
    """
    # YOUR CODE HERE
    pass


# Test
context = "The Suez Canal was opened in 1869 and connects the Red Sea to the Mediterranean."
answer = answer_with_context("When was the Suez Canal opened?", context)
print(answer)

assert answer is not None and len(answer) > 0, "Function returned empty"
assert isinstance(answer, str), "Function should return a string, not a dict"
print("\n✅ Exercise passed")

---
**Lab 03 complete.**

You understand exactly what `generate_with_single_input` does and how each parameter
changes the output. You have also written the core function of a RAG system — answering a question grounded by context.

**Next:** `lab04_llm_conversations.ipynb` — multi-turn conversations and system prompts.